# NIFTY 50 Regime Change Pipeline

Run the project in order: Phase 1 builds a local NIFTY 50 snapshot and causal features; Phase 2 fits and runs frozen PELT–L2 and PELT–RBF detectors on validation; Phase 3 compares validation labels with a training-fit, forward-filtered three-state Gaussian HMM. The final held-out PELT test cell is present but gated off by default.

**Look-ahead controls:** Features for date *t* use closes only through the previous observed session. Splits are chronological. Phase 2 fits scales, RBF bandwidth, and descriptive-volatility cutoffs on training only. The HMM is also fit on training only and uses forward filtering on validation. Do not tune settings on test. Keep each output directory new and empty.


## 0. Install dependencies

Run once in the active Jupyter environment. Restart the kernel if Jupyter asks you to.

In [ ]:
%pip install -r requirements.txt

## 1. Set paths and data source

Keep `phase1_nifty.py`, `phase2_cpd.py`, `phase3_hmm.py`, `requirements.txt`, and this notebook in the same project folder. Choose `DATA_SOURCE = "yahoo"` to download NIFTY 50 through yfinance, or set `DATA_SOURCE = "local_csv"` and provide a folder containing `NIFTY50.csv` with `Date,Close` columns. The CSV needs enough history for the chronological training, validation, and test periods.

Each run needs new output folders. Phase 3 uses the exact frozen PELT configuration (252/20, penalties 10/5). The final test cell is gated off so ordinary notebook runs leave the holdout closed.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()  # change this if the notebook is not in the project folder
PHASE1_OUTPUT = PROJECT_DIR / 'phase1_nifty50'
PHASE2_OUTPUT = PROJECT_DIR / 'phase2_nifty50_validation'
PHASE3_HMM_OUTPUT = PROJECT_DIR / 'phase3_hmm_validation'
FINAL_OUTPUT = PROJECT_DIR / 'phase2_nifty50_final_test'
DATA_SOURCE = 'yahoo'  # choose 'yahoo' or 'local_csv'
LOCAL_CSV_DIR = PROJECT_DIR / 'index_csvs'  # used only for local_csv
RUN_FINAL_TEST = False  # change to True only once, after freezing all choices

for script in ('phase1_nifty.py', 'phase2_cpd.py', 'phase3_hmm.py'):
    assert (PROJECT_DIR / script).exists(), f'{script} must be in PROJECT_DIR'
print('Project folder:', PROJECT_DIR)
print('Data source:', DATA_SOURCE)
print('Final test enabled:', RUN_FINAL_TEST)


## 2. Run Phase 1 checks

These synthetic self-tests verify the script without downloading market data.

In [ ]:
import phase1_nifty
phase1_nifty.main(['--self-test'])

## 3. Build the NIFTY 50 time series and features

This creates raw closes, aligned closes, feature rows, chronological splits, plots, and a manifest with hashes. The end date defaults to the latest completed date available to the script. Choose a new output folder for each snapshot.

In [ ]:
phase1_args = ['--indices', 'NIFTY50', '--output', str(PHASE1_OUTPUT)]
if DATA_SOURCE == 'local_csv':
    phase1_args += ['--input-dir', str(LOCAL_CSV_DIR)]
elif DATA_SOURCE != 'yahoo':
    raise ValueError("DATA_SOURCE must be 'yahoo' or 'local_csv'")

phase1_nifty.main(phase1_args)

## 4. Inspect the Phase 1 snapshot

Check that the manifest exists and that the feature file has complete rows before proceeding.

In [ ]:
manifest_path = PHASE1_OUTPUT / 'manifest.json'
features_path = PHASE1_OUTPUT / 'features' / 'NIFTY50_features.csv'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
features = pd.read_csv(features_path, parse_dates=['Date', 'source_date'])
print('Feature rows:', len(features))
print('Date range:', features['Date'].min(), 'to', features['Date'].max())
print('Rows by split:')
display(features['split'].value_counts(dropna=False).rename_axis('split').to_frame('rows'))
print('Complete rows:', int(features['complete'].fillna(False).astype(bool).sum()))
print('Manifest source:', manifest.get('source'))
display(features.head())

## 5. Run Phase 2 checks

This verifies sequential detection behavior on synthetic series. It does not use your market data.

In [ ]:
import phase2_cpd
phase2_cpd.main(['--self-test'])

## 6. Run as-of change-point detection on validation

This uses the frozen PELT settings and excludes the test split. The decision log records when an alert could have been issued; the estimated break date is kept separately. Use a new, empty output directory for each experiment.


In [ ]:
phase2_args = [
    '--input', str(PHASE1_OUTPUT),
    '--indices', 'NIFTY50',
    '--output', str(PHASE2_OUTPUT),
    '--window', '252',
    '--min-size', '20',
    '--jump', '5',
    '--every', '5',
    '--confirmations', '2',
    '--tolerance', '5',
    '--max-age', '60',
    '--cooldown', '20',
    '--penalty-l2', '10.0',
    '--penalty-rbf', '5.0',
]
phase2_cpd.main(phase2_args)


## 7. Review Phase 2 validation outputs

The output folder contains decision logs, confirmed event logs, training-only calibration details, validation timeline plots, a summary, and a run manifest. Verify `test_enabled` is false and the manifest reports window 252 and min_size 20. Do not use test results to tune parameters.


In [ ]:
summary_path = PHASE2_OUTPUT / 'summary.csv'
summary = pd.read_csv(summary_path)
display(summary)
print('Output files:')
for path in sorted(PHASE2_OUTPUT.iterdir()):
    print(path.name)

from IPython.display import Image, display
for model in ('l2', 'rbf'):
    display(Image(filename=str(PHASE2_OUTPUT / f'NIFTY50_{model}_timeline.png')))


## 8. Phase 3: fit HMM on training and compare on validation

This fits a three-state diagonal Gaussian HMM on training rows only, using the Phase 1 training-only scaler and `log_return` and `vol_20`. Ten seeded initializations are compared using training likelihood only. Validation labels use recursive forward filtering with the training filter state carried forward; there is no backward smoothing or test use. The script also verifies that the Phase 2 baseline used the frozen 252/20 configuration and excluded test data.

Use a new, empty `PHASE3_HMM_OUTPUT`. The HMM is a validation benchmark only; it is not applied to the held-out test period.


In [ ]:
import phase3_hmm

phase3_hmm.main([
    '--input', str(PHASE1_OUTPUT),
    '--phase2', str(PHASE2_OUTPUT),
    '--output', str(PHASE3_HMM_OUTPUT),
    '--states', '3',
    '--restarts', '10',
    '--seed', '1701',
])


### Review HMM comparison and validation timelines

Agreement and transition proximity compare unsupervised methods; they are not accuracy against known regimes. Review the table and chart together.


In [ ]:
hmm_comparison = pd.read_csv(PHASE3_HMM_OUTPUT / 'hmm_vs_pelt_validation.csv')
display(hmm_comparison)

hmm_validation = pd.read_csv(
    PHASE3_HMM_OUTPUT / 'NIFTY50_hmm_validation_probabilities.csv',
    parse_dates=['Date'],
)
label_order = {'calm': 0, 'moderate': 1, 'stressed': 2}
fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
for label in ('calm', 'moderate', 'stressed'):
    axes[0].plot(hmm_validation['Date'], hmm_validation[f'prob_{label}'], label=label)
axes[0].set_ylabel('HMM filtered probability')
axes[0].legend(ncol=3)
for ax, model in zip(axes[1:], ('l2', 'rbf')):
    pelt = pd.read_csv(PHASE2_OUTPUT / f'NIFTY50_{model}_decisions.csv', parse_dates=['Date'])
    joined = hmm_validation[['Date', 'hmm_regime']].merge(
        pelt[['Date', 'regime']], on='Date', how='inner', validate='one_to_one'
    )
    ax.step(joined['Date'], joined['hmm_regime'].map(label_order), where='post', label='HMM')
    ax.step(joined['Date'], joined['regime'].map(label_order), where='post', label=f'PELT-{model.upper()}')
    ax.set_yticks([0, 1, 2], ['calm', 'moderate', 'stressed'])
    ax.set_ylabel('Regime')
    ax.legend(ncol=2)
for ax in axes:
    ax.grid(alpha=.2)
axes[-1].set_xlabel('Validation date')
fig.tight_layout()
plt.show()


## 9. Optional final test: open the held-out period once

The test run for the recorded Phase 1 snapshot has already been completed. Leave `RUN_FINAL_TEST = False` when reviewing or re-running validation. For a new, deliberate final evaluation on a new frozen snapshot, first confirm all design choices are fixed, then set `RUN_FINAL_TEST = True`, use a new empty `FINAL_OUTPUT`, and execute this cell once. Do not use test results to tune the detector.


In [ ]:
if RUN_FINAL_TEST:
    phase2_cpd.main([
        '--input', str(PHASE1_OUTPUT),
        '--indices', 'NIFTY50',
        '--output', str(FINAL_OUTPUT),
        '--window', '252',
        '--min-size', '20',
        '--jump', '5',
        '--every', '5',
        '--confirmations', '2',
        '--tolerance', '5',
        '--max-age', '60',
        '--cooldown', '20',
        '--penalty-l2', '10.0',
        '--penalty-rbf', '5.0',
        '--include-test',
    ])
    final_summary = pd.read_csv(FINAL_OUTPUT / 'summary.csv')
    display(final_summary)
    from IPython.display import Image, display
    for model in ('l2', 'rbf'):
        display(Image(filename=str(FINAL_OUTPUT / f'NIFTY50_{model}_timeline.png')))
else:
    print('Held-out test remains closed. Set RUN_FINAL_TEST=True only for a deliberate one-time final run.')
